# SD-210 Synthetic Data Generator
**Sprint 3 — synthetic-data operations analytics**
This example creates and validates seven synthetic network-planning tables. All records are fictional and educational.



## 1. Configure the example
Keep the seed and simulation assumptions together so the example is easy to review and adjust.



In [36]:
from pathlib import Path

import numpy as np
import pandas as pd

SEED, FORECAST_YEARS = 210, range(4)
SITES_PER_REGION = 2
REGIONS = [
    ('R01', 'Metro North', 'urban', 0.085, 0.68),
    ('R02', 'Metro South', 'urban', 0.075, 0.67),
    ('R03', 'Lake District', 'suburban', 0.065, 0.66),
    ('R04', 'Coastal Belt', 'suburban', 0.072, 0.65),
    ('R05', 'Highland Route', 'rural', 0.048, 0.63),
    ('R06', 'River Corridor', 'rural', 0.055, 0.64),
]
PROJECT_OPTIONS = [
    ('no_action', None, 1.0, 1.0),
    ('capacity_expansion', (85000, 155001), 1.38, 0.92),
    ('targeted_densification', (145000, 245001), 1.58, 0.78),
]
if '__file__' in globals():
    OUTPUT_DIR = Path(__file__).resolve().parent.parent / 'data'
elif Path.cwd().name == 'Notebook1':
    OUTPUT_DIR = Path.cwd().parent / 'data'
elif Path.cwd().name == 'Sprint3':
    OUTPUT_DIR = Path.cwd() / 'data'
else:
    OUTPUT_DIR = Path.cwd() / 'Sprint3' / 'data'
rng = np.random.default_rng(SEED)



## 2. Create site, coverage, subscriber, QoE, traffic, project, and milestone rows
Each site's rows are made together so the seeded random draws stay in the same order on every run. Random variation represents uncertain real-world conditions in this teaching example.



In [37]:
site_rows, coverage_rows, subscriber_rows = [], [], []
qoe_rows, traffic_rows, project_rows, milestone_rows = [], [], [], []
site_number = 1



In [38]:
for region_id, region_name, market_type, growth, gross_margin in REGIONS:
    for local_site in range(SITES_PER_REGION):
        site_id = f'SITE-{site_number:03d}'
        grid_id = f'GRID-{site_number:03d}'
        # Seeded draws add realistic variation but stay repeatable.
        capacity, utilization = float(rng.uniform(2.4, 4.6)), float(rng.uniform(0.72, 1.03))
        outage_hours = float(rng.uniform(0.4, 3.0))
        if site_id == 'SITE-001':
            capacity, utilization, outage_hours = 3.2, 0.43, 0.15
        subscribers = int(rng.integers(3600, 9201))
        at_risk_share, arpu = float(rng.uniform(0.18, 0.38)), float(rng.uniform(38, 76))
        demand = capacity * utilization
        population = int(round(subscribers / 0.72))
        area = round(4.0 + site_number * 0.35, 2)
        site_rows.append((site_id, region_id, f'{region_name} site {local_site + 1}', market_type, capacity, growth, gross_margin))
        coverage_rows.append((grid_id, site_id, region_id, region_name, market_type, area, population))
        subscriber_rows.append((grid_id, site_id, subscribers, round(population / area, 2), at_risk_share, arpu))
        baseline_qoe = np.clip(100 - 50 * max(demand / capacity - 0.65, 0) - 2.5 * outage_hours, 35, 100)
        qoe_rows.append((f'QOE-{site_number:03d}', site_id, 0, round(demand * 1000, 1), round(18 + demand * 2.5 + outage_hours, 2), round(max(0.05, (100 - baseline_qoe) * 0.015), 3), outage_hours, float(baseline_qoe)))
        for year in FORECAST_YEARS:
            traffic_rows.append((site_id, year, round(demand * (1 + growth) ** year, 3)))
        for project_type, capex_range, capacity_factor, outage_factor in PROJECT_OPTIONS:
            capex = 0.0 if capex_range is None else float(rng.integers(*capex_range))
            project_id = f'{site_id}-{project_type.upper()}'
            project_rows.append((project_id, site_id, project_type, capex, capacity_factor, outage_factor))
            if project_type != 'no_action':
                for month, milestone in [(1, 'Planning'), (3, 'Permitting'), (6, 'Construction and commissioning')]:
                    milestone_rows.append((f'{project_id}-M{month:02d}', project_id, milestone, month, 'Planned'))
        site_number += 1



## 3. Assemble and export the seven menu tables
These names and columns define the generated tables.



In [39]:
TABLE_COLUMNS = {
    'CellSites': ['cell_site_id', 'region_id', 'site_name', 'market_type', 'capacity_gbps', 'annual_demand_growth', 'gross_margin'],
    'CoverageGrids': ['coverage_grid_id', 'cell_site_id', 'region_id', 'region_name', 'market_type', 'coverage_area_sq_km', 'population_covered'],
    'SubscriberDensity': ['coverage_grid_id', 'cell_site_id', 'subscriber_count', 'population_density_per_sq_km', 'at_risk_share', 'monthly_arpu_usd'],
    'QoEMeasurements': ['qoe_measurement_id', 'cell_site_id', 'measurement_year', 'download_mbps', 'latency_ms', 'packet_loss_pct', 'monthly_outage_hours', 'baseline_qoe_score'],
    'TrafficLoads': ['cell_site_id', 'forecast_year', 'peak_demand_gbps'],
    'CapexProjects': ['capex_project_id', 'cell_site_id', 'project_type', 'capex_usd', 'capacity_multiplier', 'outage_multiplier'],
    'RolloutMilestones': ['milestone_id', 'capex_project_id', 'milestone_name', 'planned_month', 'status'],
}



In [40]:
tables = {
    'CellSites': pd.DataFrame(site_rows, columns=TABLE_COLUMNS['CellSites']),
    'CoverageGrids': pd.DataFrame(coverage_rows, columns=TABLE_COLUMNS['CoverageGrids']),
    'SubscriberDensity': pd.DataFrame(subscriber_rows, columns=TABLE_COLUMNS['SubscriberDensity']),
    'QoEMeasurements': pd.DataFrame(qoe_rows, columns=TABLE_COLUMNS['QoEMeasurements']),
    'TrafficLoads': pd.DataFrame(traffic_rows, columns=TABLE_COLUMNS['TrafficLoads']),
    'CapexProjects': pd.DataFrame(project_rows, columns=TABLE_COLUMNS['CapexProjects']),
    'RolloutMilestones': pd.DataFrame(milestone_rows, columns=TABLE_COLUMNS['RolloutMilestones']),
}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, table in tables.items():
    table.to_csv(OUTPUT_DIR / f'{name}.csv', index=False)
print(f'Generated {len(tables)} synthetic SD-210 tables.')



Generated 7 synthetic SD-210 tables.


## 4. Check the exported CSV tables
Read the files back and check their schemas, row counts, identifiers, links, investment choices, and deliberate low-risk site.



In [41]:
EXPECTED_TABLES = set(TABLE_COLUMNS)
assert set(tables) == EXPECTED_TABLES
assert {path.stem for path in OUTPUT_DIR.glob('*.csv')} == EXPECTED_TABLES
exported = {name: pd.read_csv(OUTPUT_DIR / f'{name}.csv') for name in sorted(EXPECTED_TABLES)}
for name, frame in exported.items():
    assert list(frame.columns) == TABLE_COLUMNS[name], f'{name}: columns changed'
    assert len(frame) == len(tables[name]), f'{name}: row count changed'
    pd.testing.assert_frame_equal(frame, tables[name], check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-12)



In [42]:
sites, grids = exported['CellSites'], exported['CoverageGrids']
density, qoe = exported['SubscriberDensity'], exported['QoEMeasurements']
traffic, projects = exported['TrafficLoads'], exported['CapexProjects']
milestones = exported['RolloutMilestones']
assert len(sites) == 12 and sites['cell_site_id'].is_unique
assert grids['cell_site_id'].isin(sites['cell_site_id']).all() and grids['cell_site_id'].nunique() == len(sites)
assert density['coverage_grid_id'].isin(grids['coverage_grid_id']).all()
assert density['cell_site_id'].isin(sites['cell_site_id']).all()
assert qoe['cell_site_id'].isin(sites['cell_site_id']).all() and qoe['cell_site_id'].is_unique



In [43]:
assert traffic['cell_site_id'].isin(sites['cell_site_id']).all()
assert set(traffic['forecast_year']) == set(FORECAST_YEARS)
assert traffic.groupby('cell_site_id').size().eq(len(FORECAST_YEARS)).all()
assert projects['cell_site_id'].isin(sites['cell_site_id']).all()
assert projects.groupby('cell_site_id').size().eq(len(PROJECT_OPTIONS)).all()
assert (projects.loc[projects['project_type'] == 'no_action', 'capex_usd'] == 0).all()
assert milestones['capex_project_id'].isin(projects['capex_project_id']).all()
assert np.isclose(sites.loc[sites['cell_site_id'] == 'SITE-001', 'capacity_gbps'].iloc[0], 3.2)
print('PASS: all seven exported menu tables, schemas, row counts, keys, and links validated.')
pd.Series({name: len(frame) for name, frame in exported.items()}, name='rows')



PASS: all seven exported menu tables, schemas, row counts, keys, and links validated.


CapexProjects        36
CellSites            12
CoverageGrids        12
QoEMeasurements      12
RolloutMilestones    72
SubscriberDensity    12
TrafficLoads         48
Name: rows, dtype: int64

## 5. Interpretation
The tables describe a fictional network. Their internal consistency does not establish real-world engineering validity.
